In [2]:
# Check whether a GPU is available
import torch

print("GPU available:", torch.cuda.is_available())

GPU available: True


In [3]:
# Import required packages

import os
import glob
import io
import uuid
import re
import shutil
import time

from tqdm import tqdm
import numpy as np
import torch
import rawpy
from PIL import Image, UnidentifiedImageError

from flat_bug.predictor import Predictor, TensorPredictions

2026-05-22 17:13:06 - flat_bug - WARNING - _int_mm is not supported on this device, transitive closure subroutine falling back to CPU implementation


In [5]:
def parse_image(image_path, device="cpu"):
    """
    Load an image and convert it into the tensor format expected by FlatBug.

    Supported formats:
    - JPG / JPEG
    - PNG
    - DNG / RAW
    """

    if image_path.lower().endswith(".dng"):
        with rawpy.imread(image_path) as raw:
            image = Image.fromarray(raw.postprocess())
    else:
        image = Image.open(image_path)

    image = np.array(image)

    # Convert from image format (height, width, channels)
    # to PyTorch format (channels, height, width)
    return torch.from_numpy(image).permute(2, 0, 1).to(device)

In [6]:
def generate_uuid():
    """
    Generate a short unique identifier for temporary output folders.
    """
    return str(uuid.uuid4())[::3]


def wait_until_crops_finished(folder, identifier, timeout=60, stable_time=0.5):
    """
    Wait until FlatBug has finished saving all crop files. Adjust stable_time if needed.

    FlatBug saves crops asynchronously. Without this waiting step, the script
    may continue before all crop files are written, which can lead to missing
    crops in the final output.
    """

    pattern = os.path.join(folder, f"crop*{identifier}.png")

    start = time.time()
    last_count = -1
    last_change = time.time()

    while time.time() - start < timeout:
        files = glob.glob(pattern)
        count = len(files)

        if count != last_count:
            last_count = count
            last_change = time.time()

        if time.time() - last_change >= stable_time:
            return sorted(files)

        time.sleep(0.2)

    return sorted(glob.glob(pattern))

In [7]:
class Localizer(Predictor):
    """
    Wrapper around the FlatBug Predictor.

    This class runs FlatBug on images, saves the detected organisms as crops,
    and returns the crop paths for later renaming and collection.
    """

    def predict(self, images, do_plot=False, include_crops=False, outdir="output"):
        data = {
            "uuids": [],
            "predictions": [],
            "crops": [],
            "visualizations": [],
            "paths": [],
        }

        bad_images = []

        if not isinstance(images, (list, tuple)):
            images = [images]

        for img_path in tqdm(
            images,
            desc="Localizing insects",
            unit="image",
            dynamic_ncols=True,
            leave=True,
        ):
            try:
                image_tensor = parse_image(img_path, self._device)

            except UnidentifiedImageError:
                print(f"Could not read image, skipping: {img_path}")
                bad_images.append(img_path)
                continue

            # Use original image name for crop naming
            image_identifier = os.path.splitext(os.path.basename(img_path))[0]

            # Create unique temporary output folder for this image
            identifier = generate_uuid()
            this_outdir = os.path.join(outdir, identifier)
            os.makedirs(this_outdir, exist_ok=True)

            # Run FlatBug prediction
            predictions: TensorPredictions = self.pyramid_predictions(
                image_tensor,
                img_path,
                scale_before=1,
            )

            # Save detected organisms as crops
            predictions.save_crops(
                outdir=this_outdir,
                basename=image_identifier,
                mask=True,
                identifier=identifier,
            )

            # Wait until all asynchronously saved crops are actually written
            crops = wait_until_crops_finished(this_outdir, identifier)

            data["uuids"].append(identifier)
            data["crops"].append(crops)
            data["predictions"].append(predictions.json_data)
            data["paths"].append(img_path)

        if bad_images:
            print("\nThe following images were skipped because they could not be read:")
            for path in bad_images:
                print("  ", path)

        return data

In [8]:
def batch_flatbug(input_folder, output_folder, weights_path, score_threshold=0.25):
    """
    Run FlatBug on all images in a folder.

    Final crops are saved in the output folder with names such as:
    OriginalImageName_1.png
    OriginalImageName_2.png
    OriginalImageName_3.png
    """

    # ---------------------------------------------------------------------
    # 1) Collect all image files
    # ---------------------------------------------------------------------

    image_paths = []

    for root, _, files in os.walk(input_folder):
        for fname in files:
            if fname.lower().endswith((".jpg", ".jpeg", ".png", ".dng")):
                image_paths.append(os.path.join(root, fname))

    image_paths = sorted(image_paths)

    if not image_paths:
        raise ValueError(f"No images found in: {input_folder}")

    print(f"Found {len(image_paths)} images.")

    # ---------------------------------------------------------------------
    # 2) Prepare output folders
    # ---------------------------------------------------------------------

    os.makedirs(output_folder, exist_ok=True)

    # Temporary folder used before crops are renamed and collected
    tmp_outdir = os.path.join(output_folder, "_tmp_flatbug")
    os.makedirs(tmp_outdir, exist_ok=True)

    # ---------------------------------------------------------------------
    # 3) Set up device and model
    # ---------------------------------------------------------------------

    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16

    print("Using device:", device)

    model = Localizer(model=weights_path, device=device, dtype=dtype)

    model.set_hyperparameters(
        SCORE_THRESHOLD=score_threshold,
        EDGE_CASE_MARGIN=32,
        MIN_MAX_OBJ_SIZE=(16, 768),
        TIME=False,
    )

    # ---------------------------------------------------------------------
    # 4) Run FlatBug detection
    # ---------------------------------------------------------------------

    results = model.predict(
        images=image_paths,
        include_crops=False,
        outdir=tmp_outdir,
    )

    # ---------------------------------------------------------------------
    # 5) Move and rename crop files
    # ---------------------------------------------------------------------

    total_crops = 0

    for original_image, crop_paths in zip(image_paths, results["crops"]):
        base = os.path.splitext(os.path.basename(original_image))[0]

        for idx, crop_path in enumerate(crop_paths, start=1):
            if not os.path.isfile(crop_path):
                continue

            new_name = f"{base}_{idx}.png"
            target_path = os.path.join(output_folder, new_name)

            shutil.move(crop_path, target_path)
            total_crops += 1

    # ---------------------------------------------------------------------
    # 6) Clean up temporary folder
    # ---------------------------------------------------------------------

    shutil.rmtree(tmp_outdir, ignore_errors=True)

    print("DONE!")
    print(f"Processed images: {len(image_paths)}")
    print(f"Saved crops: {total_crops}")

In [9]:
# -------------------------------------------------------------------------
# User settings
# -------------------------------------------------------------------------

input_folder = r"G:\Session_1"
output_folder = r"G:\output_S1"
weights_path = r"G:\models\flatbug_weights.pt"

batch_flatbug(
    input_folder=input_folder,
    output_folder=output_folder,
    weights_path=weights_path,
    score_threshold=0.22,
)

Found 15 images.
Using device: cuda:0
YOLOv8m-seg summary (fused): 105 layers, 27,222,963 parameters, 0 gradients, 104.3 GFLOPs


Localizing insects: 100%|███████████████████████████████████████████████████████████| 15/15 [01:05<00:00,  4.37s/image]

DONE!
Processed images: 15
Saved crops: 78
